# P2｜BEATs Native Encoder

**状态：Design / Not Ready；8/20 core。** 目的：在 P1 的 joint-native reference 中，只把 ICBHI、SPRSound、KAUH 的 pooled/native encoder 从 AST 换成 BEATs；HF 固定 native temporal reference 不变。

## 唯一变量、匹配对照与四数据集边界

matched comparator=P1。唯一变量是 non-HF pooled/native encoder AST→BEATs；P2 同样固定 frozen pretrained encoder + trainable shared projector 768→256 + dataset-native heads，HF fixed temporal reference 独立且不进入 projector。输入窗口、projector 架构、native heads、HF reference、split/group、source-proportional sampler、trainable scope、update budget、seed=20260728、selection 和 metrics 必须完全匹配。

single-source comparator 继续使用相同架构的 projector + 对应 native head；joint 条件只让同一个 projector 接收 ICBHI、SPRSound、KAUH 三个 lanes。

- ICBHI cycle flat4 [B,4]；SPRSound event binary [B,2] 与 raw7 [B,7]；KAUH recording raw9 [B,9]，B/D/E 同 patient group。
- HF 仍走固定 native temporal reference；missing/unknown/gap 不是 negative。
- 输入/输出：16 kHz 音频 → BEATs pooled [B,768] → shared projected [B,256] → 三条 non-HF native logits；HF 输出与 P1 同合同。P2 不是覆盖 HF 的 shared BEATs 实验。

In [ ]:
from pathlib import Path
import os

PIPELINE = {
    "id": "P2",
    "comparator": "P1",
    "only_change": "non_hf_encoder: AST_to_BEATs",
    "seed": 20260728,
    "split_policy": "reuse_P1_immutable_receipts",
    "encoder_scope": "frozen_pretrained_BEATs",
    "shared_projector": "Linear_768_to_256_match_P1",
    "shared_projector_lanes": ["ICBHI", "SPRSound", "KAUH"],
    "sampler": "source_proportional_match_P1",
    "hf_lane": "fixed_native_temporal_reference",
    "hf_uses_shared_projector": False,
    "trainable_scope": "shared_projector_768_to_256_plus_dataset_native_heads_match_P1",
    "update_budget": None,
    "selection": None,
    "output_dir": "result/reproduce/P2_beats_native_encoder",
    "receipt_path": "result/reproduce/P2_beats_native_encoder/P2_receipt.json",
}
PROJECT_ROOT = Path(os.environ.get("ACOUSTIC_PROJECT_ROOT", Path.cwd())).resolve()
APPROVAL_RECEIPT = os.environ.get("P2_APPROVAL_RECEIPT")


## 科学 gate 与运行前审批

P1 receipt 与 AST/BEATs single-source 同架构 projector+head receipts 必须先可验；BEATs checkpoint revision/SHA、frontend、pooled contract 与 input adapter 必须锁定。配置 verifier 要证明除 non-HF encoder identity 外零差异，并冻结单一 update budget 与 selection。缺失 approval、hash 或 parity receipt 时 fail closed；不得提前读取 outer/test。

In [ ]:
required = [PIPELINE["update_budget"], PIPELINE["selection"], APPROVAL_RECEIPT]
if any(value in (None, "") for value in required):
    raise RuntimeError("P2 fail closed: matched budget, selection, and approval are not frozen")
approval_path = PROJECT_ROOT / APPROVAL_RECEIPT
if not approval_path.is_file():
    raise FileNotFoundError(approval_path)
DRY_RUN_PLAN = {"pipeline": PIPELINE, "approval": str(approval_path), "execute": False}


## 输出、receipt 与结果表

receipt 增加 P1 parity hash、BEATs checkpoint/frontend/input-adapter hashes、per-lane counts、native metrics 与 independent verifier。结果必须按任务报告，不能 pooled。

| Comparison | Native task | Result | Decision |
|---|---|---:|---|
| P2−P1 | ICBHI / SPRSound / KAUH；HF 固定 | Not run | 未判定 |

**Test Result = Not run。Decision = Not made。Claim boundary：只能归因于 non-HF package 中 AST→BEATs 的替换。**